<a href="https://colab.research.google.com/github/aavarela/SPBD_Labs/blob/main/projeto2/report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SPBD 2526 Project 2

version 0.1 (27 Nov 2025)

# Context
The project scenario involves a dataset of taxi rides, collected circa 2013, in the New York city area.

This project scenario is inspired by the ACM DEBS 2015 Grand Challenge.

### Taxi Rides

Each completed taxi ride comprises a number of attributes, separated by commas, as follows:

| Attribute   | Description |
| :---        |        :--- |
|medallion| an md5sum of the identifier of the taxi - vehicle bound|
|hack_license| an md5sum of the identifier for the taxi license|
|pickup_datetime| time when the passenger(s) were picked up|
|dropoff_datetime| time when the passenger(s) were dropped off|
|trip_time_in_secs| duration of the trip|
|trip_distance| trip distance in miles|
|pickup_longitude| longitude coordinate of the pickup location|
|pickup_latitude| latitude coordinate of the pickup location|
|dropoff_longitude| longitude coordinate of the drop-off location|
|dropoff_latitude| latitude coordinate of the drop-off location|
|payment_type| the payment method - credit card or cash|
|fare_amount| fare amount in dollars|
|surcharge| surcharge in dollars|
|mta_tax| tax in dollars|
|tip_amount| tip in dollars|
|tolls_amount| bridge and tunnel tolls in dollars|
|total_amount| total paid amount in dollars|

## Real-time Stream

For this assignment the data is presented as a "real-time" stream, made available through Apache Kafka, with the following considerations:

The stream is the same data as the original, but the timestamps (pickup_datetime and dropoff_datime) are adjusted to reflect the current time, not the date of acquisition (2013).

The stream can be replayed faster than realtime, by supplying a **speedup factor**. For example, a speedup of 60, means that 1 second in realtime, corresponds to 60 virtual seconds in the dataset.

The trip duration is not affected by the speedup factor; the same applies to the difference between dropoff_datime and pickup_datetime.


An example of the JSON encoding of each taxi ride event is the following:
```json
{"medallion": "045C16AC567D6B720C793C156F480CC4", "hack_license": "D39A22C155B4C912D0D09039BF3892B1", "pickup_datetime": "2025-11-27 16:01:19.871522", "dropoff_datetime": "2025-11-27 16:16:19.871522", "trip_time_in_secs": 900, "trip_distance": 3.59, "pickup_longitude": -74.013298, "pickup_latitude": 40.703938, "dropoff_longitude": -74.00193, "dropoff_latitude": 40.739403, "payment_type": "CRD", "fare_amount": 14.0, "surcharge": 0.5, "mta_tax": 0.5, "tip_amount": 2.9, "tolls_amount": 0.0, "total_amount": 17.9}
```

#Objectives
The main objective of this second assignment is to revisit this dataset considering that, in addition, to the original archived data, we have a realtime stream of what is happening now.

Some broad approaches to pursue in this second assignment can be:

How could the analysis selected and presented in the first assignment be refined and/or enriched by including realtime knowlege.

Can you think of an application/scenario that might be benefit or made possible by knowing what is happening now, compared to is normal in the archived dataset.

How does what is happening now, compare to what is present in the archived dataset, possibly, when evaluated in some abstract way in terms of statistical indicators (without an application in mind).

This second assignment can be seen as complementary to the first assignment. As such, it is meant as a shorter and more focused effort. It is not necessary to repeat explanations. Delivery will comprise a single notebook. The bulk of the contents should be on discussing the direction/approach being pursued and presenting any results and their associated code. Details regarding AI usage should be provided as an Addendum at the end of the notebook.

#Requeriments
Code will need to leverage Spark Structured Streaming, in some way.

#Report

In [29]:
#@title Download dataset

!wget -q -O taxi_rides_1pc.csv.gz https://www.dropbox.com/scl/fi/v8ei5laqcalrx30z3lsty/taxi_rides_1pc.csv.gz?rlkey=q1lq7l56c4j97h9kymsdroau5&st=iurdwnwj&dl=0

# Task
Set up and verify the installation of Apache Kafka and its dependency, ZooKeeper, by installing Java, downloading and extracting Kafka, then starting and checking the status of both ZooKeeper and the Kafka broker.

## Install Java and Download Kafka

### Subtask:
Install the Java Development Kit (JDK) which is a prerequisite for Kafka, and then download and extract the Apache Kafka binary distribution.


**Reasoning**:
First, I will install OpenJDK 11 using `apt-get` command to fulfill the first instruction of the subtask.



In [30]:
print("Installing OpenJDK 11...")
!sudo apt-get update
!sudo apt-get install -y openjdk-11-jdk
print("OpenJDK 11 installed.")

Installing OpenJDK 11...
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,633 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,205 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,849 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,539 kB

**Reasoning**:
Now that Java is installed, the next step is to download the Kafka binary distribution from the Apache website using `wget`. I will download a stable release and save it as `kafka.tgz`.



In [31]:
print("Downloading Apache Kafka...")
!wget -q -O kafka.tgz https://downloads.apache.org/kafka/3.7.2/kafka_2.12-3.7.2.tgz
print("Apache Kafka downloaded.")

Apache Kafka downloaded.


**Reasoning**:
With the Kafka tarball downloaded, the next step is to extract its contents using the `tar` command, as specified in the instructions. This will make the Kafka binaries and configuration files accessible.



In [32]:
print("Extracting Apache Kafka...")
!tar -xzf kafka.tgz
print("Apache Kafka extracted.")

Extracting Apache Kafka...
Apache Kafka extracted.


**Reasoning**:
Now that Kafka has been extracted, I need to change the current working directory to the extracted Kafka directory as per the instructions. This will allow subsequent Kafka commands to be executed from the correct location.



In [33]:
print("Changing directory to Kafka installation...")
%cd kafka_2.12-3.7.2
print("Changed directory to: ", !pwd)

SyntaxError: invalid syntax (ipython-input-3583583934.py, line 3)

**Reasoning**:
The previous code failed because the shell command `!pwd` cannot be directly embedded within a Python `print` statement. To fix this, I will separate the `!pwd` command from the `print` function, allowing it to execute as a standalone shell command after changing the directory.



In [34]:
print("Changing directory to Kafka installation...")
%cd kafka_2.12-3.7.2
print("Confirmed new directory:")
!pwd

Changing directory to Kafka installation...
/content/kafka_2.12-3.7.2
Confirmed new directory:
/content/kafka_2.12-3.7.2


## Start ZooKeeper

### Subtask:
Launch the ZooKeeper server, which is required by Kafka for cluster management.


**Reasoning**:
To launch the ZooKeeper server in the background and capture its output for monitoring, I will use `nohup` with the specified command, redirecting both standard output and standard error to `nohup-zookeeper.out`. A brief pause (`sleep`) will then be included to allow ZooKeeper to initialize before subsequent operations.



In [35]:
print("Starting ZooKeeper server...")
!nohup ./bin/zookeeper-server-start.sh config/zookeeper.properties > nohup-zookeeper.out 2>&1 &
!sleep 5 # Give ZooKeeper some time to start
print("ZooKeeper server started in background. Check nohup-zookeeper.out for logs.")

Starting ZooKeeper server...
ZooKeeper server started in background. Check nohup-zookeeper.out for logs.


## Start Kafka Broker

### Subtask:
Launch the Kafka broker server, which handles data streaming and storage.


**Reasoning**:
I need to start the Kafka broker server using nohup, redirecting its output to a log file, and then pause to ensure it has time to initialize, as specified in the instructions.



In [36]:
print("Starting Kafka server...")
!nohup ./bin/kafka-server-start.sh config/server.properties > nohup-kafka.out 2>&1 &
!sleep 10 # Give Kafka some time to start
print("Kafka server started in background. Check nohup-kafka.out for logs.")

Starting Kafka server...
Kafka server started in background. Check nohup-kafka.out for logs.


## Verify Kafka Status

### Subtask:
Check if Kafka and ZooKeeper are running correctly.


## Summary:

### Data Analysis Key Findings
*   OpenJDK 11 was successfully installed as a prerequisite for Kafka.
*   Apache Kafka version 3.7.2 was downloaded, extracted, and the working directory was set to its installation path, `/content/kafka_2.12-3.7.2`.
*   The ZooKeeper server was successfully started in the background, with its logs directed to `nohup-zookeeper.out`.
*   The Kafka broker server was successfully started in the background, with its logs directed to `nohup-kafka.out`.

### Insights or Next Steps
*   The essential next step is to verify the operational status of both ZooKeeper and the Kafka broker, as the current process only covered installation and background startup.
*   Future troubleshooting or monitoring can be performed by inspecting the `nohup-zookeeper.out` and `nohup-kafka.out` files for any errors or status messages.


# Task
Verify the operational status of the ZooKeeper and Kafka broker servers by using `jps` to list running Java processes and confirming the presence of the `QuorumPeerMain` and `Kafka` processes.

## Verify Kafka Status

### Subtask:
Check if Kafka and ZooKeeper are running correctly.


**Reasoning**:
To verify if ZooKeeper and Kafka are running, I will execute the `jps` command, which lists all running Java processes. This command should show `QuorumPeerMain` for ZooKeeper and `Kafka` for the Kafka broker if they are active.



In [37]:
print("Checking running Java processes...")
!jps
print("Please verify 'QuorumPeerMain' (ZooKeeper) and 'Kafka' (Kafka broker) are in the output.")

Checking running Java processes...
2389 Kafka
18887 Jps
17049 QuorumPeerMain
363 SparkSubmit
Please verify 'QuorumPeerMain' (ZooKeeper) and 'Kafka' (Kafka broker) are in the output.


## Define Data Schema

### Subtask:
Define the schema for the incoming taxi ride JSON data so Spark Structured Streaming can correctly parse and interpret the fields. This is crucial for structured processing.


**Reasoning**:
To define the data schema for Spark Structured Streaming, I need to import the necessary data types from `pyspark.sql.types` first.



In [38]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType, FloatType

print("Spark SQL data types imported.")

Spark SQL data types imported.


**Reasoning**:
Now that the necessary PySpark SQL data types are imported, I will define the `taxi_ride_schema` using `StructType` and `StructField` to match the structure and types of the incoming taxi ride JSON data, marking all fields as nullable as specified.



In [39]:
taxi_ride_schema = StructType([
    StructField("medallion", StringType(), True),
    StructField("hack_license", StringType(), True),
    StructField("pickup_datetime", TimestampType(), True),
    StructField("dropoff_datetime", TimestampType(), True),
    StructField("trip_time_in_secs", IntegerType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("pickup_longitude", DoubleType(), True),
    StructField("pickup_latitude", DoubleType(), True),
    StructField("dropoff_longitude", DoubleType(), True),
    StructField("dropoff_latitude", DoubleType(), True),
    StructField("payment_type", StringType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("surcharge", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True)
])

print("Taxi ride schema defined successfully.")

Taxi ride schema defined successfully.


## Configure Spark Structured Streaming for Kafka

### Subtask:
Set up a Spark Structured Streaming job to read the JSON messages from the Kafka topic, applying the defined schema to structure the incoming data for analysis.


**Reasoning**:
To configure Spark Structured Streaming for Kafka, I first need to import `SparkSession` and `functions` from `pyspark.sql`. Then, I will initialize a `SparkSession` with the necessary Kafka package, define the Kafka topic, and create a streaming DataFrame by reading from Kafka, parsing the JSON messages using the previously defined `taxi_ride_schema`.



In [7]:
pip install findspark

In [2]:
import findspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType, FloatType

def dumpBatchDF(df, epoch_id):
    df.show(20, False)

print("Imported necessary modules.")

# Initialize findspark to set SPARK_HOME
findspark.init()
print("Findspark initialized.")

# Stop any existing SparkSession to ensure a clean start with new configurations
if 'spark' in locals() and spark.sparkContext._jsc is not None:
    print("Stopping existing SparkSession for re-initialization...")
    spark.stop()

# Re-define schema (as SparkSession restart might affect variable scope in some environments)
taxi_ride_schema = StructType([
    StructField("medallion", StringType(), True),
    StructField("hack_license", StringType(), True),
    StructField("pickup_datetime", TimestampType(), True),
    StructField("dropoff_datetime", TimestampType(), True),
    StructField("trip_time_in_secs", IntegerType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("pickup_longitude", DoubleType(), True),
    StructField("pickup_latitude", DoubleType(), True),
    StructField("dropoff_longitude", DoubleType(), True),
    StructField("dropoff_latitude", DoubleType(), True),
    StructField("payment_type", StringType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("surcharge", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True)
])

print("Taxi ride schema defined successfully.")

# Initialize a SparkSession with the necessary Kafka JARs using spark.jars.packages
# Using version 3.3.0 for spark-sql-kafka-0-10_2.12 to match pyspark==3.3.0 and ensure Scala compatibility.
print("Initializing SparkSession with spark-sql-kafka-0-10_2.12:3.3.0...")
spark = SparkSession.builder \
    .appName("KafkaSparkStreaming") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0") \
    .getOrCreate()

print("SparkSession initialized.")

lines = spark \
  .readStream \
  .format('kafka') \
  .option('kafka.bootstrap.servers', 'localhost:9092') \
  .option('subscribe', 'taxis_json') \
  .option('startingOffsets', 'earliest') \
  .load() \
  .selectExpr('CAST(value AS STRING)')

query = lines \
    .writeStream \
    .outputMode('append') \
    .foreachBatch(dumpBatchDF) \
    .start()

query.awaitTermination(600)
query.stop()
spark.stop()



Imported necessary modules.
Findspark initialized.
Stopping existing SparkSession for re-initialization...
Taxi ride schema defined successfully.
Initializing SparkSession with spark-sql-kafka-0-10_2.12:3.3.0...
SparkSession initialized.


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: reentrant call inside <_io.BufferedReader name=49>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 511, in sen

Py4JError: An error occurred while calling o94.awaitTermination

In [ ]:
# Define the Kafka topic name
kf_topic = "taxi_rides"

# Define Kafka bootstrap servers explicitly
bootstrap_servers = ["localhost:9092"]

print(f"Kafka topic defined as: {kf_topic}")

# Use spark.readStream to create a streaming DataFrame
kf_streaming_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", ",".join(bootstrap_servers)) \
    .option("subscribe", kf_topic) \
    .load()

print("Kafka streaming DataFrame created.")

# Select the 'value' column and cast it to STRING
json_streaming_df = kf_streaming_df.selectExpr("CAST(value AS STRING)")

# Apply the taxi_ride_schema to parse the JSON 'value' column
parsed_streaming_df = json_streaming_df.select(F.from_json(F.col("value"), taxi_ride_schema).alias("data"))

# Select all fields from the parsed structured data
final_streaming_df = parsed_streaming_df.select("data.*")

print("Final streaming DataFrame with parsed JSON data created.")
print("Schema of the final streaming DataFrame:")
final_streaming_df.printSchema()